# parameter_manager — Typed Configuration Management for ML Projects

> **Combine the best of Hydra (YAML composition, `${}` interpolation, output management) with tyro (type-safe dataclasses, IDE autocompletion).**

This notebook walks through every feature of `parameter_manager` — a single-file Python library (~1500 lines) with PyYAML as its only dependency.

## 1. Quickstart

Define your config as a typed Python `@dataclass`. Load with CLI overrides or YAML files. Get a fully typed config object back.

In [ ]:
from dataclasses import dataclass, field
from parameter_manager import load_config, to_plain, render_help

@dataclass
class TrainConfig:
    """My training configuration."""
    lr: float = 3e-4
    batch_size: int = 32
    exp_name: str = "default"
    seed: int = 42

# Pure defaults — no YAML, no CLI
cfg = load_config(TrainConfig)
print(f"lr={cfg.lr}, batch_size={cfg.batch_size}, exp_name={cfg.exp_name}")
print(f"type: {type(cfg).__name__}")  # TrainConfig — real typed object!

In [ ]:
# Override from code (or CLI: --lr=1e-4 --exp_name=my_run)
cfg = load_config(TrainConfig, overrides=["lr=1e-4", "exp_name=my_run"])
print(f"lr={cfg.lr}, exp_name={cfg.exp_name}")

In [ ]:
# See the full config schema
print(render_help(TrainConfig))

## 2. Nested Dataclasses

Nested configs mirror your code hierarchy. Override nested fields with dotted paths.

In [ ]:
@dataclass
class ModelConfig:
    hidden_dim: int = 256
    num_layers: int = 6
    dropout: float = 0.1

@dataclass
class OptimizerConfig:
    name: str = "adam"
    lr: float = 3e-4
    weight_decay: float = 0.01

@dataclass
class FullConfig:
    model: ModelConfig = field(default_factory=ModelConfig)
    optimizer: OptimizerConfig = field(default_factory=OptimizerConfig)
    max_epochs: int = 100

# Override nested fields with dotted paths
cfg = load_config(FullConfig, overrides=[
    "model.hidden_dim=512",
    "optimizer.lr=1e-4",
    "max_epochs=200"
])
print(f"model.hidden_dim={cfg.model.hidden_dim}")
print(f"optimizer.lr={cfg.optimizer.lr}")
print(f"max_epochs={cfg.max_epochs}")
# Unchanged defaults still work
print(f"model.num_layers={cfg.model.num_layers}")

In [ ]:
# Nested render_help
print(render_help(FullConfig))

## 3. YAML File Loading

Load one or more YAML files. Later files override earlier ones. Fields not in the YAML keep their dataclass defaults.

In [ ]:
import tempfile, os

tmp = tempfile.mkdtemp()

# Write a YAML config file
with open(os.path.join(tmp, "experiment.yaml"), "w") as f:
    f.write("""
model:
  hidden_dim: 512
  num_layers: 12
optimizer:
  name: adamw
  lr: 1e-4
max_epochs: 200
""")

cfg = load_config(FullConfig, config_files=[os.path.join(tmp, "experiment.yaml")])
print(f"model.hidden_dim = {cfg.model.hidden_dim}")   # from YAML: 512
print(f"optimizer.name = {cfg.optimizer.name}")        # from YAML: adamw
print(f"model.dropout = {cfg.model.dropout}")          # from default: 0.1

In [ ]:
# Precedence: dataclass defaults < YAML file 1 < YAML file 2 < CLI overrides

# Write two YAML files — base and override
with open(os.path.join(tmp, "base.yaml"), "w") as f:
    f.write("model:\n  hidden_dim: 128\nmax_epochs: 50\n")

with open(os.path.join(tmp, "override.yaml"), "w") as f:
    f.write("model:\n  hidden_dim: 1024\n")

# Later files win: override.yaml beats base.yaml for hidden_dim
# CLI beats everything for max_epochs
cfg = load_config(
    FullConfig,
    config_files=[
        os.path.join(tmp, "base.yaml"),
        os.path.join(tmp, "override.yaml")
    ],
    overrides=["max_epochs=999"]
)
print(f"model.hidden_dim = {cfg.model.hidden_dim}")  # 1024 (from override.yaml)
print(f"max_epochs = {cfg.max_epochs}")              # 999 (from CLI)

## 4. Config Groups (Hydra-style `defaults:`)

YAML files can declare a `defaults:` list to compose config fragments — swap database backends, model architectures, etc.

In [ ]:
# Create a config tree:
#   conf/
#     base.yaml          # defaults: [{db: mysql}]
#     db/
#       mysql.yaml       # host: localhost, port: 3306
#       postgres.yaml    # host: pg.example.com, port: 5432

conf_dir = os.path.join(tmp, "conf")
os.makedirs(os.path.join(conf_dir, "db"))

with open(os.path.join(conf_dir, "base.yaml"), "w") as f:
    f.write("""
defaults:
  - db: mysql          # load conf/db/mysql.yaml → merge under 'db' key
lr: 3e-4
exp_name: group_demo
""")

with open(os.path.join(conf_dir, "db", "mysql.yaml"), "w") as f:
    f.write("host: localhost\nport: 3306\n")

with open(os.path.join(conf_dir, "db", "postgres.yaml"), "w") as f:
    f.write("host: pg.example.com\nport: 5432\n")

# Define matching dataclass
@dataclass
class DBConfig:
    host: str = ""
    port: int = 0

@dataclass
class AppConfig:
    db: DBConfig = field(default_factory=DBConfig)
    lr: float = 1e-3
    exp_name: str = "default"

cfg = load_config(AppConfig, config_files=[os.path.join(conf_dir, "base.yaml")])
print(f"db.host = {cfg.db.host}")       # localhost (from db/mysql.yaml)
print(f"db.port = {cfg.db.port}")       # 3306
print(f"lr = {cfg.lr}")                 # 3e-4 (from base.yaml)
print(f"exp_name = {cfg.exp_name}")     # group_demo

In [ ]:
# Swap the group option at load-time by changing the YAML reference
with open(os.path.join(conf_dir, "base.yaml"), "w") as f:
    f.write("""
defaults:
  - db: postgres       # switch to postgres!
lr: 3e-4
""")

cfg = load_config(AppConfig, config_files=[os.path.join(conf_dir, "base.yaml")])
print(f"db.host = {cfg.db.host}")  # pg.example.com
print(f"db.port = {cfg.db.port}")  # 5432

## 5. CLI Override Grammar

The library extracts `--dotted.path=value` tokens from `sys.argv`. Everything else (`--flag`, positional args) passes through for your own argument parser.

In [ ]:
from parameter_manager import parse_overrides

# Simulate CLI: python train.py --model.hidden_dim=512 --verbose input.txt
overrides, rest = parse_overrides([
    "--model.hidden_dim=512",
    "--lr=1e-3",
    "--verbose",           # not consumed — no '=' sign
    "input.txt"             # not consumed — positional
])
print(f"overrides: {overrides}")
print(f"rest (for your own parser): {rest}")

In [ ]:
# Typo protection — difflib suggestions
try:
    load_config(TrainConfig, overrides=["batch_szie=64"])  # misspelled!
except Exception as e:
    print(f"Error: {e}")

## 6. Interpolation Engine

Values containing `${...}` are resolved automatically. Supported expressions:

| Expression | Meaning | Example |
|---|---|---|
| `${a.b.c}` | Config cross-reference (absolute path) | `${model.hidden_dim}` |
| `${env:NAME}` | Environment variable | `${env:HOME}` |
| `${env:NAME:def}` | Env var with default | `${env:CUDA_VISIBLE_DEVICES:0}` |
| `${now:FORMAT}` | `strftime` (frozen per load) | `${now:%Y-%m-%d_%H-%M-%S}` |
| `${now}` | ISO-8601 timestamp | `${now}` |
| `${eval:EXPR}` | Python expression | `${eval:cfg.lr * 10}` |
| `$${` | Escaped literal `${` | `"$${not_interp}"` |

In [ ]:
# Cross-reference interpolation with type preservation
@dataclass
class InterpDemo:
    hidden_dim: int = 256
    embed_dim: int = 0      # will reference hidden_dim
    exp_name: str = "demo"
    log_dir: str = ""        # combines cross-ref + now

cfg = load_config(InterpDemo, overrides=[
    "embed_dim=${hidden_dim}",              # type-preserving: int → int
    "log_dir=logs/${exp_name}/${now:%Y-%m-%d}"  # multi-interpolation
])
print(f"embed_dim = {cfg.embed_dim}  (type: {type(cfg.embed_dim).__name__})")
print(f"log_dir = {cfg.log_dir}")

In [ ]:
# Environment variable interpolation
import os
os.environ["MY_USER"] = "alice"

@dataclass
class EnvDemo:
    user: str = ""
    gpu: str = ""

cfg = load_config(EnvDemo, overrides=[
    "user=${env:MY_USER}",
    "gpu=${env:CUDA_VISIBLE_DEVICES:0}"  # falls back to "0" if env var missing
])
print(f"user = {cfg.user}")
print(f"gpu = {cfg.gpu}")

In [ ]:
# Eval interpolation — compute derived values
@dataclass
class EvalDemo:
    base_lr: float = 3e-4
    scaled_lr: float = 0.0
    total_steps: int = 1000
    warmup_steps: int = 0

cfg = load_config(EvalDemo, overrides=[
    "scaled_lr=${eval:cfg.base_lr * math.sqrt(8)}",
    "warmup_steps=${eval:int(cfg.total_steps * 0.1)}"
])
print(f"scaled_lr = {cfg.scaled_lr:.6f}")
print(f"warmup_steps = {cfg.warmup_steps}")

In [ ]:
# Cycle detection — prevents infinite loops
@dataclass
class CycleDemo:
    x: str = ""
    y: str = ""

try:
    load_config(CycleDemo, overrides=["x=${y}", "y=${x}"])
except Exception as e:
    print(f"Cycle caught: {e}")

In [ ]:
# Escaping: $${ → literal ${
@dataclass
class EscapeDemo:
    template: str = ""

cfg = load_config(EscapeDemo, overrides=["template=This $${literal} is not resolved"])
print(cfg.template)

## 7. Type Conversion Reference

All values from YAML and CLI overrides are converted to the declared field type. Invalid conversions raise clear errors.

In [ ]:
from typing import Optional, List, Dict, Any, Literal
from enum import Enum
from pathlib import Path

class Scheduler(Enum):
    COSINE = "cosine"
    LINEAR = "linear"
    CONSTANT = "constant"

@dataclass
class TypeDemo:
    # Scalars
    flag: bool = True
    count: int = 42
    rate: float = 0.001
    
    # Enum / Literal
    scheduler: Scheduler = Scheduler.COSINE
    precision: Literal["fp16", "fp32", "bf16"] = "fp32"
    
    # Path / Optional
    data_dir: Path = Path("./data")
    wandb_project: Optional[str] = None
    
    # Containers
    layer_dims: List[int] = field(default_factory=lambda: [256, 128, 64])
    tags: Dict[str, Any] = field(default_factory=dict)

cfg = load_config(TypeDemo, overrides=[
    "flag=false",
    "count=99",
    "rate=0.5",
    "scheduler=LINEAR",
    "precision=fp16",
    "data_dir=/mnt/data",
    "wandb_project=my_project",
    "layer_dims=512,256,128",     # comma-separated list
    'tags={"env":"prod","ver":2}' # JSON dict
])

print(f"flag={cfg.flag}, count={cfg.count}, rate={cfg.rate}")
print(f"scheduler={cfg.scheduler}, precision={cfg.precision}")
print(f"data_dir={cfg.data_dir}, wandb_project={cfg.wandb_project}")
print(f"layer_dims={cfg.layer_dims}, tags={cfg.tags}")

In [ ]:
# Type errors give clear, path-specific messages
try:
    load_config(TypeDemo, overrides=["count=not_a_number"])
except Exception as e:
    print(f"Error: {e}")

## 8. Required Fields

Fields without defaults are **required**. They must be supplied by YAML or CLI overrides.

In [ ]:
@dataclass
class RequiredDemo:
    dataset_path: str        # no default → REQUIRED
    batch_size: int = 32     # has default → optional

# Without override: error
try:
    load_config(RequiredDemo)
except Exception as e:
    print(f"Error: {e}")

# With override: works
cfg = load_config(RequiredDemo, overrides=["dataset_path=/data/imagenet"])
print(f"dataset_path = {cfg.dataset_path}")

## 9. Output Management

When `save=True` and `output_dir` is set, the resolved config is saved alongside metadata — perfect for experiment tracking.

In [ ]:
import parameter_manager as pm

@dataclass
class OutputDemo:
    lr: float = 3e-4
    batch_size: int = 32
    exp_name: str = "demo_output"
    output_dir: str = "outputs/${exp_name}/${now:%Y-%m-%d_%H-%M-%S}"

cfg = load_config(OutputDemo, output_dir="outputs/${exp_name}", save=True)

if pm.last_output_dir:
    print(f"Saved to: {pm.last_output_dir}")
    for f in sorted(pm.last_output_dir.iterdir()):
        print(f"  {f.name}")

In [ ]:
# Read back the saved config
import yaml
if pm.last_output_dir:
    with open(pm.last_output_dir / "config.yaml") as f:
        saved = yaml.safe_load(f)
    print("Saved config:")
    for k, v in saved.items():
        print(f"  {k}: {v}")

In [ ]:
# Read meta.json
import json
if pm.last_output_dir:
    with open(pm.last_output_dir / "meta.json") as f:
        meta = json.load(f)
    print("Meta:")
    for k, v in meta.items():
        print(f"  {k}: {v}")

In [ ]:
# Round-trip: load_config from saved config dicts (no YAML file needed)
if pm.last_output_dir:
    with open(pm.last_output_dir / "config.yaml") as f:
        saved_dict = yaml.safe_load(f)
    cfg2 = load_config(OutputDemo, config_files=[saved_dict])
    print(f"Round-trip OK: lr={cfg2.lr}, batch_size={cfg2.batch_size}")

## 10. Real-World ML Pipeline

Putting it all together: a realistic training config with nested models, optimizers, data, logging, and output management.

In [ ]:
@dataclass
class ModelConfig:
    """Model architecture."""
    hidden_dim: int = 256
    num_layers: int = 6
    num_heads: int = 8
    dropout: float = 0.1
    activation: Literal["relu", "gelu", "silu"] = "gelu"

@dataclass
class OptimizerConfig:
    """Optimizer settings."""
    name: Literal["adam", "adamw", "sgd"] = "adamw"
    lr: float = 3e-4
    weight_decay: float = 0.01
    betas: List[float] = field(default_factory=lambda: [0.9, 0.999])

@dataclass
class DataConfig:
    """Data loading."""
    path: str = "./data"
    batch_size: int = 32
    num_workers: int = 4
    shuffle: bool = True
    augment: bool = True

@dataclass
class TrainConfig:
    """Top-level training configuration."""
    model: ModelConfig = field(default_factory=ModelConfig)
    optimizer: OptimizerConfig = field(default_factory=OptimizerConfig)
    data: DataConfig = field(default_factory=DataConfig)
    exp_name: str = "default"
    seed: int = 42
    max_steps: int = 10000
    eval_every: int = 500
    log_dir: str = "logs/${exp_name}/${now:%Y-%m-%d_%H-%M-%S}"

# Load from YAML + CLI overrides
cfg = load_config(TrainConfig, overrides=[
    "model.hidden_dim=512",
    "model.num_layers=12",
    "optimizer.lr=1e-4",
    "data.batch_size=64",
    "exp_name=large_transformer",
    "seed=123"
])

print(f"Experiment: {cfg.exp_name}")
print(f"Model: {cfg.model.num_layers} layers, dim={cfg.model.hidden_dim}")
print(f"Optimizer: {cfg.optimizer.name}, lr={cfg.optimizer.lr}")
print(f"Data: batch={cfg.data.batch_size}, workers={cfg.data.num_workers}")
print(f"Logging to: {cfg.log_dir}")
print(f"Seed: {cfg.seed}")

In [ ]:
# The entire config is inspectable as a plain dict
import json
print(json.dumps(to_plain(cfg), indent=2, default=str))

## Summary

| Feature | How |
|---|---|
| **Config schema** | Standard `@dataclass` — full IDE support |
| **YAML files** | Load & merge multiple files with `config_files=[...]` |
| **Config groups** | `defaults: [{db: mysql}]` in YAML — Hydra-style composition |
| **CLI overrides** | `--model.hidden_dim=512` — non-hijacking (rest passes through) |
| **Interpolation** | `${path}`, `${env:VAR}`, `${now:...}`, `${eval:...}` |
| **Type validation** | All values converted & validated against field types |
| **Output** | `config.yaml` + `meta.json` + `overrides.txt` per run |
| **Dependencies** | PyYAML only (~1.5k lines, single file) |

---

**vs Hydra**: typed configs (not DictConfig), no ConfigStore, no CLI hijacking, minimal deps.

**vs tyro**: YAML file loading, `${}` interpolation, config group composition, output management.